# Diseño del Pipeline de Calidad de Datos

## Objetivo

Implementar un pipeline de análisis de calidad de datos sobre los datasets **TLC Trip Record Data** siguiendo la arquitectura **Medallion**, permitiendo evaluar automáticamente la calidad de los datos consolidados de la capa Silver mediante ocho dimensiones de calidad.

El pipeline será completamente desarrollado en **PySpark**, tendrá procesamiento **incremental**, generará **metadata**, **logs de auditoría** y almacenará los resultados en formatos **Parquet** y **JSONL** para su posterior consumo desde Power BI.

---

# Ubicación dentro de la Arquitectura

```
data/
│
├── bronze/
│
├── silver/
│   ├── consolidated/
│   │   ├── trip_data/
│   │   ├── lookup/
│   │   └── _metadata/
│   │
│   └── quality/
│       ├── quality_metrics.parquet
│       ├── quality_metrics.jsonl
│       └── _metadata/
│
├── gold/
│
└── logs/
    ├── tlc_ingestion_manifest.jsonl
    ├── silver_manifest.jsonl
    └── quality_manifest.jsonl
```

---

# Ubicación del Script

```
src/
│
├── ingestion/
│
├── silver/
│   ├── consolidacion_por_categoria.py
│   └── analisis_calidad_datos.py
│
├── gold/
├── audit/
└── utils/
```

El análisis de calidad pertenece a la **capa Silver**, ya que evalúa la calidad de los datos consolidados antes de ser utilizados por los modelos analíticos y dashboards.

---

# Flujo del Pipeline

```
Silver Consolidated
        │
        ▼
Leer archivos Parquet
        │
        ▼
¿El archivo ya fue analizado?
        │
   Sí ─────────────► Omitir
        │
        ▼
        No
        │
        ▼
Ejecutar análisis
de calidad
        │
        ▼
Generar métricas
        │
        ▼
Actualizar
quality_metrics.parquet
        │
        ▼
Actualizar
quality_metrics.jsonl
        │
        ▼
Generar metadata
        │
        ▼
Agregar registro al
quality_manifest.jsonl
```

---

# Procesamiento Incremental

El pipeline seguirá exactamente la misma filosofía implementada en la ingesta y en la consolidación Silver.

## Primera ejecución

Procesará todos los datasets disponibles.

Ejemplo:

- Yellow 2023
- Yellow 2024
- Yellow 2025
- Green 2023
- Green 2024
- Green 2025
- FHV
- FHVHV

---

## Segunda ejecución

Si únicamente aparece:

```
yellow_2026.parquet
```

el pipeline solamente analizará ese nuevo archivo.

Los años 2023, 2024 y 2025 serán omitidos porque ya fueron procesados anteriormente.

---

# Dimensiones de Calidad

Se evaluarán las siguientes ocho dimensiones.

---

## 1. Completitud

Evalúa la existencia de valores faltantes.

### Reglas

- NULL
- NaN
- cadenas vacías

### Métrica

```
Completitud =
1 - (Valores faltantes / Total de valores)
```

---

## 2. Exactitud

Evalúa que los valores numéricos sean correctos.

### Ejemplos

- trip_distance ≥ 0
- fare_amount ≥ 0
- tip_amount ≥ 0
- total_amount ≥ 0
- passenger_count ≥ 0

---

## 3. Consistencia

Evalúa relaciones lógicas entre columnas.

### Ejemplos

pickup_datetime < dropoff_datetime

total_amount ≥ fare_amount

trip_distance = 0
→ total_amount debería ser bajo

---

## 4. Integridad

Evalúa integridad referencial utilizando el catálogo Taxi Zone Lookup.

### Reglas

PULocationID existe en taxi_zone_lookup

DOLocationID existe en taxi_zone_lookup

---

## 5. Razonabilidad

Detecta valores improbables.

Ejemplos

trip_distance < 200 millas

passenger_count ≤ 8

tip_amount ≤ total_amount

Duración del viaje < 24 horas

---

## 6. Oportunidad

Comprueba que el año del archivo coincida con el año contenido en los registros.

Ejemplo

```
yellow_2025.parquet
```

Todos los viajes deben pertenecer al año 2025.

---

## 7. Unicidad

Detecta registros duplicados.

Se analizarán:

Duplicados completos

Duplicados utilizando:

- VendorID
- Pickup Datetime
- Dropoff Datetime
- PULocationID
- DOLocationID

---

## 8. Validez

Verifica que los datos pertenezcan a sus dominios permitidos.

Ejemplos

VendorID

```
1
2
```

payment_type

```
1
2
3
4
5
6
```

RateCodeID

```
1
2
3
4
5
6
99
```

Store_and_fwd_flag

```
Y
N
NULL
```

---

# Salida del Pipeline

Se generarán dos archivos con exactamente la misma información.

## quality_metrics.parquet

Será utilizado por:

- Power BI
- Dashboards
- Procesos analíticos posteriores

---

## quality_metrics.jsonl

Será una representación en formato JSONL de las mismas métricas para facilitar auditoría y trazabilidad.

Ejemplo

```json
{
    "dataset":"yellow",
    "anio":2025,
    "dimension":"Completitud",
    "columna":"fare_amount",
    "problemas":24,
    "porcentaje":0.0008,
    "severidad":"BAJA"
}
```

---

# Estructura del Dataset de Calidad

Cada fila representará una regla evaluada.

|Campo|Descripción|
|------|-----------|
|dataset|Tipo de dataset|
|anio|Año analizado|
|dimension|Dimensión evaluada|
|columna|Columna analizada|
|problemas|Cantidad de registros afectados|
|porcentaje|Porcentaje afectado|
|severidad|Alta, Media o Baja|
|fecha_ejecucion|Fecha del análisis|

---

# Metadata

Por cada archivo procesado se generará una metadata independiente.

Ubicación

```
data/silver/quality/_metadata/
```

Ejemplo

```
yellow_2023.json
green_2024.json
fhv_2025.json
```

Ejemplo de contenido

```json
{
    "pipeline_id":"quality_20260711_203015",
    "pipeline_name":"quality_analysis",
    "pipeline_version":"1.0.0",
    "dataset":"yellow",
    "anio":2025,
    "archivo_origen":"yellow_2025.parquet",
    "registros_analizados":37201522,
    "dimensiones_calculadas":8,
    "quality_metrics_path":"data/silver/quality/quality_metrics.parquet",
    "processing_time_seconds":8.34,
    "generated_at":"2026-07-11T20:30:15"
}
```

La metadata permitirá identificar qué archivos ya fueron analizados para soportar el procesamiento incremental.

---

# Logs del Pipeline

Cada ejecución agregará una nueva línea al archivo

```
data/logs/quality_manifest.jsonl
```

Nunca se sobrescribirá.

Ejemplo

```json
{
    "pipeline_id":"quality_20260711_203015",
    "status":"SUCCESS",
    "files_processed":1,
    "files_skipped":12,
    "duration_seconds":8.34,
    "generated_at":"2026-07-11T20:30:15"
}
```

---

# Estructura General del Script

El archivo

```
analisis_calidad_datos.py
```

mantendrá la misma organización utilizada en el resto del proyecto.

```
Configuración

Rutas

Spark

Utilidades

Lectura incremental

Funciones de calidad

Completitud

Exactitud

Consistencia

Integridad

Razonabilidad

Oportunidad

Unicidad

Validez

Generación de métricas

Actualización incremental

Metadata

Logs

Main
```

De esta forma se conserva una estructura homogénea con los pipelines de Ingesta y Silver, facilitando el mantenimiento, la trazabilidad y la escalabilidad del proyecto.

# Diseño del Pipeline de Calidad de Datos

## Objetivo

Implementar un pipeline de análisis de calidad de datos sobre los datasets **TLC Trip Record Data** siguiendo la arquitectura **Medallion**, permitiendo evaluar automáticamente la calidad de los datos consolidados de la capa Silver mediante ocho dimensiones de calidad.

El pipeline será completamente desarrollado en **PySpark**, tendrá procesamiento **incremental**, generará **metadata**, **logs de auditoría** y almacenará los resultados en formatos **Parquet** y **JSONL** para su posterior consumo desde Power BI.

---

# Ubicación dentro de la Arquitectura

```
data/
│
├── bronze/
│
├── silver/
│   ├── consolidated/
│   │   ├── trip_data/
│   │   ├── lookup/
│   │   └── _metadata/
│   │
│   └── quality/
│       ├── quality_metrics.parquet
│       ├── quality_metrics.jsonl
│       └── _metadata/
│
├── gold/
│
└── logs/
    ├── tlc_ingestion_manifest.jsonl
    ├── silver_manifest.jsonl
    └── quality_manifest.jsonl
```

---

# Ubicación del Script

```
src/
│
├── ingestion/
│
├── silver/
│   ├── consolidacion_por_categoria.py
│   └── analisis_calidad_datos.py
│
├── gold/
├── audit/
└── utils/
```

El análisis de calidad pertenece a la **capa Silver**, ya que evalúa la calidad de los datos consolidados antes de ser utilizados por los modelos analíticos y dashboards.

---

# Flujo del Pipeline

```
Silver Consolidated
        │
        ▼
Leer archivos Parquet
        │
        ▼
¿El archivo ya fue analizado?
        │
   Sí ─────────────► Omitir
        │
        ▼
        No
        │
        ▼
Ejecutar análisis
de calidad
        │
        ▼
Generar métricas
        │
        ▼
Actualizar
quality_metrics.parquet
        │
        ▼
Actualizar
quality_metrics.jsonl
        │
        ▼
Generar metadata
        │
        ▼
Agregar registro al
quality_manifest.jsonl
```

---


# Procesamiento Incremental

El pipeline seguirá exactamente la misma filosofía implementada en la ingesta y en la consolidación Silver.

## Primera ejecución

Procesará todos los datasets disponibles.

Ejemplo:

- Yellow 2023
- Yellow 2024
- Yellow 2025
- Green 2023
- Green 2024
- Green 2025
- FHV
- FHVHV

---

## Segunda ejecución

Si únicamente aparece:

```
yellow_2026.parquet
```

el pipeline solamente analizará ese nuevo archivo.

Los años 2023, 2024 y 2025 serán omitidos porque ya fueron procesados anteriormente.

---

# Dimensiones de Calidad

Se evaluarán las siguientes ocho dimensiones.

---

## 1. Completitud

Evalúa la existencia de valores faltantes.

### Reglas

- NULL
- NaN
- cadenas vacías

### Métrica

```
Completitud =
1 - (Valores faltantes / Total de valores)
```

---

## 2. Exactitud

Evalúa que los valores numéricos sean correctos.

### Ejemplos

- trip_distance ≥ 0
- fare_amount ≥ 0
- tip_amount ≥ 0
- total_amount ≥ 0
- passenger_count ≥ 0

---

## 3. Consistencia

Evalúa relaciones lógicas entre columnas.

### Ejemplos

pickup_datetime < dropoff_datetime

total_amount ≥ fare_amount

trip_distance = 0
→ total_amount debería ser bajo

---

## 4. Integridad

Evalúa integridad referencial utilizando el catálogo Taxi Zone Lookup.

### Reglas

PULocationID existe en taxi_zone_lookup

DOLocationID existe en taxi_zone_lookup

---

## 5. Razonabilidad

Detecta valores improbables.

Ejemplos

trip_distance < 200 millas

passenger_count ≤ 8

tip_amount ≤ total_amount

Duración del viaje < 24 horas

---

## 6. Oportunidad

Comprueba que el año del archivo coincida con el año contenido en los registros.

Ejemplo

```
yellow_2025.parquet
```

Todos los viajes deben pertenecer al año 2025.

---

## 7. Unicidad

Detecta registros duplicados.

Se analizarán:

Duplicados completos

Duplicados utilizando:

- VendorID
- Pickup Datetime
- Dropoff Datetime
- PULocationID
- DOLocationID

---

## 8. Validez

Verifica que los datos pertenezcan a sus dominios permitidos.

Ejemplos

VendorID

```
1
2
```

payment_type

```
1
2
3
4
5
6
```

RateCodeID

```
1
2
3
4
5
6
99
```

Store_and_fwd_flag

```
Y
N
NULL
```

---

# Salida del Pipeline

Se generarán dos archivos con exactamente la misma información.

## quality_metrics.parquet

Será utilizado por:

- Power BI
- Dashboards
- Procesos analíticos posteriores

---

## quality_metrics.jsonl

Será una representación en formato JSONL de las mismas métricas para facilitar auditoría y trazabilidad.

Ejemplo

```json
{
    "dataset":"yellow",
    "anio":2025,
    "dimension":"Completitud",
    "columna":"fare_amount",
    "problemas":24,
    "porcentaje":0.0008,
    "severidad":"BAJA"
}
```

---

# Estructura del Dataset de Calidad

Cada fila representará una regla evaluada.

|Campo|Descripción|
|------|-----------|
|dataset|Tipo de dataset|
|anio|Año analizado|
|dimension|Dimensión evaluada|
|columna|Columna analizada|
|problemas|Cantidad de registros afectados|
|porcentaje|Porcentaje afectado|
|severidad|Alta, Media o Baja|
|fecha_ejecucion|Fecha del análisis|

---

# Metadata

Por cada archivo procesado se generará una metadata independiente.

Ubicación

```
data/silver/quality/_metadata/
```

Ejemplo

```
yellow_2023.json
green_2024.json
fhv_2025.json
```

Ejemplo de contenido

```json
{
    "pipeline_id":"quality_20260711_203015",
    "pipeline_name":"quality_analysis",
    "pipeline_version":"1.0.0",
    "dataset":"yellow",
    "anio":2025,
    "archivo_origen":"yellow_2025.parquet",
    "registros_analizados":37201522,
    "dimensiones_calculadas":8,
    "quality_metrics_path":"data/silver/quality/quality_metrics.parquet",
    "processing_time_seconds":8.34,
    "generated_at":"2026-07-11T20:30:15"
}
```

La metadata permitirá identificar qué archivos ya fueron analizados para soportar el procesamiento incremental.

---

# Logs del Pipeline

Cada ejecución agregará una nueva línea al archivo

```
data/logs/quality_manifest.jsonl
```

Nunca se sobrescribirá.

Ejemplo

```json
{
    "pipeline_id":"quality_20260711_203015",
    "status":"SUCCESS",
    "files_processed":1,
    "files_skipped":12,
    "duration_seconds":8.34,
    "generated_at":"2026-07-11T20:30:15"
}
```

---

# Estructura General del Script

El archivo

```
analisis_calidad_datos.py
```

mantendrá la misma organización utilizada en el resto del proyecto.

```
Configuración

Rutas

Spark

Utilidades

Lectura incremental

Funciones de calidad

Completitud

Exactitud

Consistencia

Integridad

Razonabilidad

Oportunidad

Unicidad

Validez

Generación de métricas

Actualización incremental

Metadata

Logs

Main
```

De esta forma se conserva una estructura homogénea con los pipelines de Ingesta y Silver, facilitando el mantenimiento, la trazabilidad y la escalabilidad del proyecto.